In [6]:
import torch
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer
from datasets import load_dataset
from tqdm import tqdm

from harm_model import HarmScoringModel

# ---------------- CONFIG ----------------
DEVICE = torch.device("cuda")
BASELINE_PATH = "baseline_model.pt"
NEW_MODEL_PATH = "transfer_learning_model.pt"
BATCH_SIZE = 16
MAX_SAMPLES = 500
# ---------------------------------------
class InferenceDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.texts = dataframe["prompt"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }

# 1. Load JailbreakV-28K
ds = load_dataset("JailbreakV-28K/JailBreakV-28k", "JailBreakV_28K")
df = ds["JailBreakV_28K"].to_pandas()

# 2. Extract prompt text
df["prompt"] = df["redteam_query"]

# 3. Sample fixed 500 prompts
df_sample = df.sample(
    n=min(MAX_SAMPLES, len(df)),
    random_state=42
).reset_index(drop=True)

# 4. Tokenizer + Dataset
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
dataset = InferenceDataset(df_sample, tokenizer)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

# 5. Load models
baseline_model = HarmScoringModel().to(DEVICE)
baseline_model.load_state_dict(torch.load(BASELINE_PATH, map_location=DEVICE))
baseline_model.eval()

new_model = HarmScoringModel().to(DEVICE)
new_model.load_state_dict(torch.load(NEW_MODEL_PATH, map_location=DEVICE))
new_model.eval()

# 6. Run inference
baseline_scores = []
new_scores = []

with torch.no_grad():
    for batch in tqdm(loader, desc="Running inference"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        base_logits = baseline_model(input_ids, attention_mask)
        new_logits = new_model(input_ids, attention_mask)

        baseline_scores.extend(torch.sigmoid(base_logits).cpu().numpy())
        new_scores.extend(torch.sigmoid(new_logits).cpu().numpy())

# 7. Save results
df_sample["baseline_score"] = baseline_scores
df_sample["new_model_score"] = new_scores

df_sample[["prompt", "baseline_score", "new_model_score"]].to_csv(
    "jailbreak_model_comparison.csv",
    index=False
)

print("Saved to jailbreak_model_comparison.csv")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\T2430392\AppData\Local\Temp\ipykernel_98196\798735168.py:58: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_

Saved to jailbreak_model_comparison.csv


In [8]:
import pandas as pd

# Load results
df = pd.read_csv("jailbreak_model_comparison.csv")

# Ensure numeric
df["baseline_score"] = pd.to_numeric(df["baseline_score"], errors="coerce")
df["new_model_score"] = pd.to_numeric(df["new_model_score"], errors="coerce")

# 1. New model score > baseline score
better_count = (df["new_model_score"] > df["baseline_score"]).sum()

# 2. Scores below threshold (0.7)
baseline_above_06 = (df["baseline_score"] > 0.6).sum()
new_above_06 = (df["new_model_score"] > 0.6).sum()

# Print results
print(f"Rows where new model > baseline: {better_count}")
print(f"Baseline scores > 0.6: {baseline_above_06}")
print(f"New model scores > 0.6: {new_above_06}")

Rows where new model > baseline: 314
Baseline scores > 0.6: 170
New model scores > 0.6: 190


In [9]:
import torch
import pandas as pd
from torch.utils.data import DataLoader
from transformers import RobertaTokenizer
from tqdm import tqdm

from harm_model import HarmScoringModel

# ---------------- CONFIG ----------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "transfer_learning_model.pt"
BATCH_SIZE = 16
MAX_LENGTH = 128
INPUT_CSV = "KDDataset.csv"
OUTPUT_CSV = "KD_Dataset_with_harmscore.csv"
# ---------------------------------------


class InferenceDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.texts = dataframe["prompt"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0)
        }


# 1. Load KD dataset
df = pd.read_csv(INPUT_CSV)

# 2. Build unified dataframe
normal_df = df[["normal_prompts"]].dropna().rename(
    columns={"normal_prompts": "prompt"}
)
normal_df["type"] = "normal"

jailbreak_df = df[["jailbreak_prompts"]].dropna().rename(
    columns={"jailbreak_prompts": "prompt"}
)
jailbreak_df["type"] = "jailbreak"

final_df = pd.concat([normal_df, jailbreak_df], ignore_index=True)

# 3. Tokenizer + DataLoader
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
dataset = InferenceDataset(final_df, tokenizer, MAX_LENGTH)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

# 4. Load transfer model
model = HarmScoringModel().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

# 5. Run inference
harmscores = []

with torch.no_grad():
    for batch in tqdm(loader, desc="Running harm scoring"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        logits = model(input_ids, attention_mask)
        scores = torch.sigmoid(logits).cpu().numpy()

        harmscores.extend(scores)

# 6. Save results
final_df["harmscore"] = harmscores

final_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved results to {OUTPUT_CSV}")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\T2430392\AppData\Local\Temp\ipykernel_98196\3994832252.py:65: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe

Saved results to KD_Dataset_with_harmscore.csv
